# Tutorial: Brainclip Colab Voice Runtime

This notebook creates an isolated virtual environment inside Colab, installs pinned dependencies, starts a FastAPI voice runtime, and exposes it through Cloudflare Tunnel.


## Why this notebook uses a virtual environment

Colab runtime packages change over time. Brainclip pins versions in a local venv so package drift in the base image does not silently break the voice server.


In [ ]:
from pathlib import Path
import os
import sys

APP_ROOT = Path('/content')
VENV_DIR = APP_ROOT / 'venv'
print('Python version:', sys.version)
if sys.version_info < (3, 10):
    raise RuntimeError('Brainclip runtime expects Python 3.10 or newer.')


In [ ]:
%%bash
set -e
cd /content
if [ ! -d "notebooks" ]; then
  git clone https://github.com/harshal0704/Brainclip.git temp_repo
  mv temp_repo/* .
  rm -rf temp_repo
fi
python -m venv /content/venv
source /content/venv/bin/activate
python -m pip install --upgrade 'pip<25' 'setuptools<75' 'wheel<0.46'
python -m pip install -r /content/notebooks/requirements.txt


In [ ]:
%%bash
set -e
source /content/venv/bin/activate
export BRAINCLIP_APP_ROOT=/content
python -m uvicorn notebooks.brainclip_colab_server:app --app-dir /content --host 0.0.0.0 --port 8000 > /content/uvicorn.log 2>&1 &
sleep 5
python -m pip install cloudflared >/dev/null 2>&1 || true
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
cloudflared tunnel --url http://127.0.0.1:8000 --logfile /content/cloudflared.log > /content/cloudflared.out 2>&1 &
sleep 8
grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' /content/cloudflared.out | head -n 1


## Operational notes

- Paste the printed Cloudflare URL into Brainclip settings as the Colab tunnel URL.
- The runtime now fetches the latest scripts directly from the Brainclip repository.
